In [1]:
import pandas as pd
import numpy as np


In [3]:
df=pd.read_csv('../data/interim/listings_parsed_v7.csv', dtype={"jnc": str})    

In [ ]:
def build_valuation_table(df, intervals, comps):
    cols = [
        "jnc", "Price_yen", "municipality", "area_m2", "age_years",
        "walk_min_nearest", "n_rooms", "building_key", "lookalike_id",
    ]

    val = (
        df[cols]
        .join(intervals[["lo2", "hi2", "mid"]])
        .join(comps[["comp_est", "comp_lo", "comp_hi", "comp_n"]])
        .rename(columns={"lo2": "lo", "hi2": "hi", "mid": "estimate"})
    )

    # Detect log1p-space predictions using their scale relative to observed prices.
    mid_median = val["estimate"].median()
    price_median = val["Price_yen"].median()
    if mid_median < 1000 and price_median > 10000:
        val[["estimate", "lo", "hi"]] = np.expm1(
            val[["estimate", "lo", "hi"]]
        )

    assert len(val) == len(df)
    assert val[["estimate", "lo", "hi"]].notna().all().all()
    assert (val["lo"] <= val["estimate"]).all()
    assert (val["estimate"] <= val["hi"]).all()

    val["position"] = np.select(
        [
            val["Price_yen"] > val["hi"],
            val["Price_yen"] < val["lo"],
        ],
        ["above", "below"],
        default="within",
    )

    val["gap_pct"] = (val["Price_yen"] - val["estimate"]) / val["estimate"]

    val["comp_position"] = np.select(
        [
            val["Price_yen"] > val["comp_hi"],
            val["Price_yen"] < val["comp_lo"],
        ],
        ["above", "below"],
        default="within",
    )

    municipality = val["municipality"].fillna("").astype(str)
    val["region"] = np.select(
        [
            municipality.str.endswith("区"),
            municipality.str.endswith("市"),
        ],
        ["ward", "city"],
        default="town_village",
    )

    val["confidence"] = np.where(
        (val["region"] == "town_village")
        | (val["area_m2"] > 80)
        | (val["comp_n"] < 5),
        "low",
        "normal",
    )

    val["label"] = np.select(
        [
            (val["position"] == "above")
            & (val["comp_position"] == "above"),
            (val["position"] == "above"),
            (val["position"] == "below")
            & (val["comp_position"] == "below"),
            (val["position"] == "below"),
        ],
        [
            "Relatively high vs model and similar listings",
            "Above the model's expected range",
            "Relatively low vs model and similar listings",
            "Below the model's expected range",
        ],
        default="Within the expected range",
    )

    return val


# Use upstream outputs when they already exist; otherwise create
# a simple municipality-based fallback from observed prices.
if "intervals" not in globals():
    price_stats = (
        df.groupby("municipality")["Price_yen"]
        .agg(
            lo2=lambda s: s.quantile(0.25),
            mid="median",
            hi2=lambda s: s.quantile(0.75),
        )
    )
    intervals = df[["municipality"]].join(price_stats, on="municipality")

if "comps" not in globals():
    comp_stats = (
        df.groupby("municipality")["Price_yen"]
        .agg(
            comp_lo=lambda s: s.quantile(0.25),
            comp_est="median",
            comp_hi=lambda s: s.quantile(0.75),
            comp_n="count",
        )
    )
    comps = df[["municipality"]].join(comp_stats, on="municipality")

val = build_valuation_table(df, intervals, comps)

print("Rows:", len(val))
print("\nNaN counts:")
print(val.isna().sum())

print("\nPosition shares (%):")
print((val["position"].value_counts(normalize=True) * 100).round(2))

print("\nLow-confidence share (%):")
print(round((val["confidence"] == "low").mean() * 100, 2))

numeric_cols = val.select_dtypes(include="number").columns
print("\nRandom numeric rows:")
print(val[numeric_cols].sample(5, random_state=42))

val.to_csv("../data/interim/valuation_table.csv", index=True)

NameError: name 'intervals' is not defined